#  DSAI202 – Data Governance | Phase 1
---
Omar Hany 202301146 - Nader Mohamed 202400453 -
Malak Osama 202402421 - Rowida Sayed 202402291

## 1 ▸ Setup – Install Libraries

In [1]:
%pip install -q ydata-profiling pandera matplotlib seaborn openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Omar Darwish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [273]:
import pandas as pd
import numpy as np
import re
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandera as pa
from pandera import Column, DataFrameSchema, Check
from ydata_profiling import ProfileReport
from IPython.display import display, HTML, FileLink

## 2 ▸ Data Loading

In [274]:
import os

FILE_PATH = "Ask A Manager Salary Survey 2021 (Responses).csv"

In [275]:
df_raw = pd.read_csv(FILE_PATH)
df_raw.head(3)

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


## 3 ▸ Data Profiling

In [276]:
prof_rows = []
for col in df_raw.columns:
    s = df_raw[col]
    prof_rows.append({
        'Column'         : col,
        'dtype'          : str(s.dtype),
        'Non-Null'       : int(s.notna().sum()),
        'Null Count'     : int(s.isna().sum()),
        'Null %'         : round(s.isna().mean() * 100, 1),
        'Unique Values'  : int(s.nunique()),
        'Sample Value'   : str(s.dropna().iloc[0]) if s.notna().any() else 'N/A',
    })

profile_df = pd.DataFrame(prof_rows)
display(profile_df.to_string(index=False))

'                                                                                                                                                                                                                              Column   dtype  Non-Null  Null Count  Null %  Unique Values                       Sample Value\n                                                                                                                                                                                                                           Timestamp  object     28211           0     0.0          25429                 4/27/2021 11:02:10\n                                                                                                                                                                                                                    How old are you?  object     28211           0     0.0              7                              25-34\n                                             

In [277]:
profile = ProfileReport(
     df_raw,
     title        = 'Ask A Manager Salary Survey 2021 – Data Profiling Report',
     explorative  = True,
     minimal      = False,
 )

PROFILE_HTML = 'profiling_report.html'
profile.to_file(PROFILE_HTML)
print(f'Report saved → {PROFILE_HTML}')
display(FileLink(PROFILE_HTML))

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:08<00:00,  2.23it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report saved → profiling_report.html


c:\Users\Omar Darwish\Downloads\phase 1 governance\Submission\profiling_report.html

## 4 ▸ Data Cleaning

In [278]:
df = df_raw.copy()

df.columns = [
    'timestamp',
    'age_group',
    'industry',
    'job_title',
    'job_title_context',
    'annual_salary_raw',
    'additional_comp',
    'currency',
    'currency_other',
    'income_context',
    'country',
    'us_state',
    'city',
    'years_exp_overall',
    'years_exp_field',
    'education',
    'gender',
    'race',
]
print(df.columns.tolist())

['timestamp', 'age_group', 'industry', 'job_title', 'job_title_context', 'annual_salary_raw', 'additional_comp', 'currency', 'currency_other', 'income_context', 'country', 'us_state', 'city', 'years_exp_overall', 'years_exp_field', 'education', 'gender', 'race']


In [279]:
df['timestamp'] = pd.to_datetime(df['timestamp'], infer_datetime_format=True, errors='coerce')
print('Timestamp nulls after parse:', df['timestamp'].isna().sum())
print('Date range:', df['timestamp'].min(), '→', df['timestamp'].max())

Timestamp nulls after parse: 0
Date range: 2021-04-27 11:02:10 → 2026-03-19 18:35:16


C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\854716844.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['timestamp'] = pd.to_datetime(df['timestamp'], infer_datetime_format=True, errors='coerce')


In [280]:
def parse_salary(val):
    if pd.isna(val):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(val).replace(',', ''))
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df['annual_salary'] = df['annual_salary_raw'].apply(parse_salary)

before = len(df)
df = df[(df['annual_salary'].isna()) | ((df['annual_salary'] >= 1_000) & (df['annual_salary'] <= 5_000_000))]
print(f'Rows removed (salary out of range): {before - len(df)}')
print(df['annual_salary'].describe())

Rows removed (salary out of range): 161
count    2.805000e+04
mean     9.362283e+04
std      1.209977e+05
min      1.000000e+03
25%      5.400000e+04
50%      7.523450e+04
75%      1.100000e+05
max      5.000000e+06
Name: annual_salary, dtype: float64


In [281]:
USA_VARIANTS = [
    'united states', 'usa', 'us', 'u.s.', 'u.s.a.', 'united states of america',
    'united states of american', 'us of a', 'the united states', 'america',
    'united  states', 'unitied states', 'united statew', 'united sates',
    'united stares',
]

def standardise_country(val):
    if pd.isna(val):
        return np.nan
    low = str(val).strip().lower()
    if low in USA_VARIANTS:
        return 'United States'
    return str(val).strip().title()

df['country'] = df['country'].apply(standardise_country)
print('Top 10 countries after standardisation:')
print(df['country'].value_counts().head(10))

Top 10 countries after standardisation:
country
United States     23047
Canada             1675
Uk                  690
United Kingdom      634
Australia           390
Germany             187
England             170
Ireland             124
New Zealand         123
France               67
Name: count, dtype: int64


In [282]:
text_cols = ['industry', 'job_title', 'city', 'us_state', 'currency']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace('nan', np.nan)

df['gender'] = df['gender'].astype(str).str.strip().str.title().replace('Nan', np.nan)

df['race'] = df['race'].astype(str).str.strip().replace('nan', np.nan)

df['education'] = df['education'].astype(str).str.strip().replace('nan', np.nan)

In [283]:
EXP_MAP = {
    '1 year or less'  : 0.5,
    '2 - 4 years'     : 3.0,
    '5-7 years'       : 6.0,
    '8 - 10 years'    : 9.0,
    '11 - 20 years'   : 15.5,
    '21 - 30 years'   : 25.5,
    '31 - 40 years'   : 35.5,
    '41 years or more': 41.0,
}

df['years_exp_overall_num'] = df['years_exp_overall'].map(EXP_MAP)
df['years_exp_field_num']   = df['years_exp_field'].map(EXP_MAP)

AGE_MAP = {
    'under 18'  : 16,
    '18-24'     : 21,
    '25-34'     : 29,
    '35-44'     : 39,
    '45-54'     : 49,
    '55-64'     : 59,
    '65 or over': 67,
}
df['age_midpoint'] = df['age_group'].str.lower().str.strip().map(AGE_MAP)
print(df[['age_group','age_midpoint','years_exp_overall','years_exp_overall_num']].head(5))

  age_group  age_midpoint years_exp_overall  years_exp_overall_num
0     25-34            29         5-7 years                    6.0
1     25-34            29      8 - 10 years                    9.0
2     25-34            29       2 - 4 years                    3.0
3     25-34            29      8 - 10 years                    9.0
4     25-34            29      8 - 10 years                    9.0


In [284]:
MAJOR_CURRENCIES = ['USD','GBP','CAD','AUD','EUR','CHF','SEK','NOK','DKK','INR','NZD','HKD','SGD','ZAR','JPY','MXN','BRL']

def standardise_currency(row):
    cur = str(row['currency']).strip().upper()
    other = str(row['currency_other']).strip().upper() if pd.notna(row['currency_other']) else ''
    if cur == 'OTHER' and other in MAJOR_CURRENCIES:
        return other
    if cur in MAJOR_CURRENCIES:
        return cur
    match = re.search(r'\b([A-Z]{3})\b', cur)
    return match.group(1) if match else 'OTHER'

df['currency_clean'] = df.apply(standardise_currency, axis=1)
print('Currency distribution:')
print(df['currency_clean'].value_counts())

Currency distribution:
currency_clean
USD      23401
CAD       1671
GBP       1591
EUR        632
AUD        508
OTHER       86
SEK         38
CHF         38
ZAR         16
SGD         13
JPY         12
INR         11
DKK         11
NOK          9
BRL          5
HKD          4
NZD          2
MXN          2
Name: count, dtype: int64


In [285]:
df['additional_comp'] = df['additional_comp'].fillna(0)
df['total_comp'] = df['annual_salary'].fillna(0) + df['additional_comp']

In [286]:
before = len(df)
df = df.drop_duplicates()
print(f'Duplicate rows removed: {before - len(df)}')
print(f'Final shape: {df.shape}')

Duplicate rows removed: 0
Final shape: (28050, 24)


In [287]:
thresh = 0.95
drop_cols = [c for c in df.columns if df[c].isna().mean() > thresh]
print(f'Dropping high-null columns (>{thresh*100:.0f}% missing): {drop_cols}')
df.drop(columns=drop_cols, inplace=True)

print(f'\nRemaining null counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Dropping high-null columns (>95% missing): ['currency_other']

Remaining null counts:
industry                72
job_title                1
job_title_context    20797
income_context       25033
country                  1
us_state              4995
city                    81
education              222
gender                 175
race                   185
dtype: int64


In [288]:
CLEAN_CSV = 'salary_survey_2021_cleaned.csv'
df.to_csv(CLEAN_CSV, index=False)
display(FileLink(CLEAN_CSV))

c:\Users\Omar Darwish\Downloads\phase 1 governance\Submission\salary_survey_2021_cleaned.csv

## 5 ▸ Validation Schema (Pandera)

In [289]:
VALID_CURRENCIES = ['USD','GBP','CAD','AUD','EUR','CHF','SEK','NOK','DKK','INR',
                    'NZD','HKD','SGD','ZAR','JPY','MXN','BRL','OTHER']
VALID_AGE_GROUPS = ['under 18','18-24','25-34','35-44','45-54','55-64','65 or over']
VALID_EXP        = ['1 year or less','2 - 4 years','5-7 years','8 - 10 years',
                    '11 - 20 years','21 - 30 years','31 - 40 years','41 years or more']
VALID_EDUCATION  = [
    'High School', 'Some college', 'College degree',
    "Master's degree", 'PhD', 'JD', 'MD', 'MBA', 'Other',
    'Professional degree (MD, JD, etc.)', 'Some postgraduate work',
]

schema = DataFrameSchema(
    columns={
        'timestamp': Column(
            pa.DateTime,
            nullable=True,
            description='Survey response timestamp'
        ),
        'age_group': Column(
            pa.String,
            nullable=False,
            checks=Check.isin(VALID_AGE_GROUPS),
            description='Respondent age bracket'
        ),
        'industry': Column(
            pa.String,
            nullable=True,
            checks=Check.str_length(min_value=2, max_value=200),
            description='Industry of employment'
        ),
        'job_title': Column(
            pa.String,
            nullable=True,
            checks=Check.str_length(min_value=2, max_value=300),
            description='Job title of respondent'
        ),
        'annual_salary': Column(
            pa.Float,
            nullable=True,
            checks=[
                Check.greater_than_or_equal_to(1_000),
                Check.less_than_or_equal_to(5_000_000),
            ],
            description='Annual salary in local currency'
        ),
        'additional_comp': Column(
            pa.Float,
            nullable=False,
            checks=Check.greater_than_or_equal_to(0),
            description='Bonuses / additional monetary compensation'
        ),
        'currency_clean': Column(
            pa.String,
            nullable=False,
            checks=Check.isin(VALID_CURRENCIES),
            description='Standardised ISO 4217 currency code'
        ),
        'country': Column(
            pa.String,
            nullable=True,
            checks=Check.str_length(min_value=2, max_value=100),
            description='Country of work'
        ),
        'years_exp_overall': Column(
            pa.String,
            nullable=True,
            checks=Check.isin(VALID_EXP),
            description='Total years of professional experience'
        ),
        'years_exp_field': Column(
            pa.String,
            nullable=True,
            checks=Check.isin(VALID_EXP),
            description='Years of experience in current field'
        ),
        'education': Column(
            pa.String,
            nullable=True,
            description='Highest education level'
        ),
        'gender': Column(
            pa.String,
            nullable=True,
            description='Gender identity'
        ),
        'race': Column(
            pa.String,
            nullable=True,
            description='Racial identity (may be multiple)'
        ),
    },
    strict=False,
    coerce=True,
    name='SalarySurvey2021Schema',
)


In [290]:
try:
    validated_df = schema.validate(df, lazy=True)
    print('All rows passed schema validation!')
except pa.errors.SchemaErrors as exc:
    failure_cases = exc.failure_cases
    display(failure_cases.head(30))
    print('\nSummary by column:')
    print(failure_cases.groupby(['schema_context','column'])['failure_case'].count())

,schema_context,column,check,check_number,failure_case,index
0,Column,job_title,"str_length(2, 300)",0,f,14779
1,Column,job_title,"str_length(2, 300)",0,-,17081
2,Column,job_title,"str_length(2, 300)",0,-,17088
3,Column,country,"str_length(2, 100)",0,"We Don'T Get Raises, We Get Quarterly Bonuses,...",2249
4,Column,country,"str_length(2, 100)",0,"I Earn Commission On Sales. If I Meet Quota, I...",12925
5,Column,country,"str_length(2, 100)",0,I Was Brought In On This Salary To Help With T...,16853
6,Column,country,"str_length(2, 100)",0,Y,22659



Summary by column:
schema_context  column   
Column          country      4
                job_title    3
Name: failure_case, dtype: int64


## 6 ▸ Dashboard – Visualisations

In [291]:
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2',
           '#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']
sns.set_theme(style='whitegrid', palette=PALETTE)

In [292]:
df_usd = df[(df['currency_clean'] == 'USD') & (df['annual_salary'].notna())].copy()
print(f'USD respondents: {len(df_usd):,}')

USD respondents: 23,401


In [293]:
age_order = ['under 18','18-24','25-34','35-44','45-54','55-64','65 or over']
age_counts = df['age_group'].value_counts().reindex(age_order).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(age_counts.index, age_counts.values, color=PALETTE[:len(age_counts)], edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_title('Respondents by Age Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group', fontsize=11)
ax.set_ylabel('Number of Respondents', fontsize=11)
ax.set_ylim(0, age_counts.max() * 1.15)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_age_dist.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\2784498997.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [294]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_usd['annual_salary'].clip(upper=300_000), bins=60,
        color='#4C72B0', edgecolor='white', linewidth=0.5, alpha=0.9)
ax.axvline(df_usd['annual_salary'].median(), color='#C44E52', linewidth=2,
           linestyle='--', label=f'Median: ${df_usd["annual_salary"].median():,.0f}')
ax.axvline(df_usd['annual_salary'].mean(), color='#DD8452', linewidth=2,
           linestyle=':', label=f'Mean: ${df_usd["annual_salary"].mean():,.0f}')
ax.set_title('Annual Salary Distribution (USD respondents, clipped at $300k)', fontsize=13, fontweight='bold')
ax.set_xlabel('Annual Salary (USD)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('chart_salary_dist.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\980000602.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [295]:
age_salary = (
    df_usd.groupby('age_group')['annual_salary']
    .median()
    .reindex(age_order)
    .dropna()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(age_salary.index, age_salary.values,
              color=PALETTE[:len(age_salary)], edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='$%.0f', padding=4, fontsize=9)
ax.set_title('Median Annual Salary (USD) by Age Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group', fontsize=11)
ax.set_ylabel('Median Salary (USD)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_salary_age.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\4213612113.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [296]:
top_industries = df['industry'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top_industries.index[::-1], top_industries.values[::-1],
        color=PALETTE[:12], edgecolor='white')
for i, v in enumerate(top_industries.values[::-1]):
    ax.text(v + 30, i, f'{v:,}', va='center', fontsize=9)
ax.set_title('Top 12 Industries by Number of Respondents', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Respondents', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_industries.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\1521948383.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [297]:
top8 = df_usd['industry'].value_counts().head(8).index
df_top8 = df_usd[df_usd['industry'].isin(top8)]

fig, ax = plt.subplots(figsize=(13, 6))
df_top8.boxplot(
    column='annual_salary', by='industry', ax=ax,
    patch_artist=True, showfliers=False,
    boxprops=dict(facecolor='#4C72B0', color='#333'),
    medianprops=dict(color='#C44E52', linewidth=2),
)
ax.set_title('Salary Distribution by Industry (USD, top 8)', fontsize=13, fontweight='bold')
plt.suptitle('')
ax.set_xlabel('Industry', fontsize=11)
ax.set_ylabel('Annual Salary (USD)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.xticks(rotation=25, ha='right', fontsize=8)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_salary_industry.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\2098210767.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [298]:
gender_counts = df['gender'].value_counts().head(5)

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    gender_counts.values,
    labels=gender_counts.index,
    colors=PALETTE[:len(gender_counts)],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
)
for text in autotexts:
    text.set_fontsize(10)
ax.set_title('Gender Distribution of Respondents', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_gender.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\1556932305.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [299]:
edu_salary = (
    df_usd.groupby('education')['annual_salary']
    .agg(['median','count'])
    .query('count >= 30')
    .sort_values('median', ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(edu_salary.index, edu_salary['median'],
               color='#55A868', edgecolor='white', linewidth=0.8)
for i, (idx, row) in enumerate(edu_salary.iterrows()):
    ax.text(row['median'] + 500, i, f"${row['median']:,.0f}", va='center', fontsize=9)
ax.set_title('Median Annual Salary (USD) by Education Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Median Salary (USD)', fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_education_salary.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\3512601915.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [300]:
cur_counts = df['currency_clean'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cur_counts.index, cur_counts.values,
              color=PALETTE[:len(cur_counts)], edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_title('Top 10 Currencies Used by Respondents', fontsize=14, fontweight='bold')
ax.set_xlabel('Currency', fontsize=11)
ax.set_ylabel('Number of Respondents', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_currency.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\2330541720.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [301]:
exp_order = ['1 year or less','2 - 4 years','5-7 years','8 - 10 years',
             '11 - 20 years','21 - 30 years','31 - 40 years','41 years or more']
exp_salary = (
    df_usd.groupby('years_exp_overall')['annual_salary']
    .median()
    .reindex(exp_order)
    .dropna()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(len(exp_salary)), exp_salary.values, marker='o',
        color='#4C72B0', linewidth=2.5, markersize=8)
for i, v in enumerate(exp_salary.values):
    ax.annotate(f'${v/1000:.0f}k', (i, v), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
ax.set_xticks(range(len(exp_salary)))
ax.set_xticklabels(exp_salary.index, rotation=20, ha='right', fontsize=9)
ax.set_title('Median Annual Salary (USD) by Years of Total Experience', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Salary (USD)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_exp_salary.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\4170007725.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [302]:
country_counts = df['country'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(country_counts.index[::-1], country_counts.values[::-1],
        color='#8172B2', edgecolor='white')
for i, v in enumerate(country_counts.values[::-1]):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=9)
ax.set_title('Top 10 Countries by Number of Respondents', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Respondents', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_country.png', dpi=150)
plt.show()

C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\749365691.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8 ▸ Summary Statistics Table

In [303]:
print(f'Total records          : {len(df):,}')
print(f'Total columns          : {df.shape[1]}')
print(f'USD respondents        : {len(df_usd):,}')
print(f'Unique countries       : {df["country"].nunique()}')
print(f'Unique industries      : {df["industry"].nunique()}')
print(f'Unique job titles      : {df["job_title"].nunique()}')
print(f'\nUSD Salary Statistics:')
print(f'  Min    : ${df_usd["annual_salary"].min():>12,.0f}')
print(f'  Q1     : ${df_usd["annual_salary"].quantile(0.25):>12,.0f}')
print(f'  Median : ${df_usd["annual_salary"].median():>12,.0f}')
print(f'  Mean   : ${df_usd["annual_salary"].mean():>12,.0f}')
print(f'  Q3     : ${df_usd["annual_salary"].quantile(0.75):>12,.0f}')
print(f'  Max    : ${df_usd["annual_salary"].max():>12,.0f}')
print(f'  Std    : ${df_usd["annual_salary"].std():>12,.0f}')

Total records          : 28,050
Total columns          : 23
USD respondents        : 23,401
Unique countries       : 245
Unique industries      : 1129
Unique job titles      : 13416

USD Salary Statistics:
  Min    : $       1,000
  Q1     : $      56,238
  Median : $      78,500
  Mean   : $      92,355
  Q3     : $     112,000
  Max    : $   5,000,000
  Std    : $      75,522


# Phase 2: Privacy and Security Implementation

This notebook demonstrates Data Security, Encryption, Access Control, and Regulatory Compliance. We have divided the workflow into detailed steps with clear milestones.

## Step 1: Loading and Preparing the Dataset
First, we load our cleaned dataset from Phase 1.

### Simulating Sensitive Data
The original dataset does not contain highly sensitive PII like passwords or phone numbers. To fulfill the requirements of Phase 2, we will simulate these fields for each record.

In [ ]:
import random
import string

def generate_fake_phone():
    return f"+1-{random.randint(100, 999)}-{random.randint(100, 999)}-{random.randint(1000, 9999)}"

def generate_fake_password():
    return ''.join(random.choices(string.ascii_letters + string.digits, k=8))

print("Columns before adding sensitive data:", list(df.columns))

df['phone_number'] = [generate_fake_phone() for _ in range(len(df))]
df['password'] = [generate_fake_password() for _ in range(len(df))]

print("\nColumns after adding sensitive data:", list(df.columns))
print("\nSample of the simulated sensitive data added:")
display(df[['job_title', 'phone_number', 'password']].head())

Columns before adding sensitive data: ['timestamp', 'age_group', 'industry', 'job_title', 'job_title_context', 'annual_salary_raw', 'additional_comp', 'currency', 'income_context', 'country', 'us_state', 'city', 'years_exp_overall', 'years_exp_field', 'education', 'gender', 'race', 'annual_salary', 'years_exp_overall_num', 'years_exp_field_num', 'age_midpoint', 'currency_clean', 'total_comp']

Columns after adding sensitive data: ['timestamp', 'age_group', 'industry', 'job_title', 'job_title_context', 'annual_salary_raw', 'additional_comp', 'currency', 'income_context', 'country', 'us_state', 'city', 'years_exp_overall', 'years_exp_field', 'education', 'gender', 'race', 'annual_salary', 'years_exp_overall_num', 'years_exp_field_num', 'age_midpoint', 'currency_clean', 'total_comp', 'phone_number', 'password']

Sample of the simulated sensitive data added:


,job_title,phone_number,password
0,Research and Instruction Librarian,+1-129-875-8046,zH2I9bBD
1,Change & Internal Communications Manager,+1-231-268-7952,eBwyxoos
2,Marketing Specialist,+1-268-738-2084,ppinhBam
3,Program Manager,+1-121-456-4465,3THgf6YH
4,Accounting Manager,+1-123-169-9493,lGBH88ph


In [ ]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Omar Darwish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [306]:
import hashlib

def hash_data(data):
    return hashlib.sha256(str(data).encode()).hexdigest()

df['hashed_password'] = df['password'].apply(hash_data)
df['hashed_phone_number'] = df['phone_number'].apply(hash_data)

print("Hashing Complete! Let's compare the plain text with the hashed outputs.")

comparison_df = df[['password', 'hashed_password', 'phone_number', 'hashed_phone_number']].head()
display(comparison_df)

Hashing Complete! Let's compare the plain text with the hashed outputs.


,password,hashed_password,phone_number,hashed_phone_number
0,zH2I9bBD,57f3d3a37fb6153b490688893257ba261697c093025871...,+1-129-875-8046,3e464fb1ceee3004cfe3c26792175cf850f680e6aa960f...
1,eBwyxoos,1d427d398651386ea626f7b3fb76f34fda24290db932db...,+1-231-268-7952,a6baad9f2bacd73154741b4df4d37eb5c0b1e8c911513e...
2,ppinhBam,712ed5f73332876e4a662935023ff575b762b357733925...,+1-268-738-2084,9968c4ccfa98b13e439b59b276ec60724cbf48b2d0041b...
3,3THgf6YH,4f4804f0f7258eff188742b572b631bcce989353431829...,+1-121-456-4465,0dc059017522494340e17abcbd99daef7c20c7497e656a...
4,lGBH88ph,e4df756b4a031a38d1ff3c30f5c6502e8458cd99566c4d...,+1-123-169-9493,5affd3f98f408b12af3f82ed11a0e58f9c13348b550e46...


## Step 2: Data Security and Encryption

### 2.1 Hashing Sensitive Information
We will use `hashlib` to apply SHA-256 hashing to the newly generated passwords and phone numbers to secure them.

In [307]:
import math

def is_prime(n):
    if n <= 1: return False
    if n <= 3: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

def generate_keypair(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('Both numbers must be prime.')
    elif p == q:
        raise ValueError('p and q cannot be equal')
    
    n = p * q
    phi = (p-1) * (q-1)
    
    e = random.randrange(1, phi)
    g = math.gcd(e, phi)
    while g != 1:
        e = random.randrange(1, phi)
        g = math.gcd(e, phi)
        
    d = pow(e, -1, phi)
    return ((e, n), (d, n))

print("RSA functions defined successfully.")

RSA functions defined successfully.


### 2.2 RSA Encryption and Decryption from Scratch
Next, we build the RSA encryption algorithm from scratch to encrypt textual data.

In [308]:
def is_prime(n):
    if n <= 1: return False
    if n <= 3: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

def generate_keypair(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('Both numbers must be prime.')
    elif p == q:
        raise ValueError('p and q cannot be equal')
    
    n = p * q
    phi = (p-1) * (q-1)
    
    e = random.randrange(1, phi)
    g = math.gcd(e, phi)
    while g != 1:
        e = random.randrange(1, phi)
        g = math.gcd(e, phi)
        
    d = pow(e, -1, phi)
    return ((e, n), (d, n))

print("RSA functions defined successfully.")


RSA functions defined successfully.


In [ ]:
def rsa_encrypt(pk, plaintext):
    key, n = pk
    return [pow(ord(char), key, n) for char in plaintext]

def rsa_decrypt(pk, ciphertext):
    key, n = pk
    return ''.join([chr(pow(char, key, n)) for char in ciphertext])

p, q = 61, 53
public, private = generate_keypair(p, q)
print(f"Generated Public Key: {public}")
print(f"Generated Private Key: {private}")

sample_text = str(df['job_title'].iloc[0])
encrypted_msg = rsa_encrypt(public, sample_text)
decrypted_msg = rsa_decrypt(private, encrypted_msg)

print("\n--- RSA Test Results ---")
print("Plain Text:     ", sample_text)
print("Encrypted Text: ", encrypted_msg)
print("Decrypted Text: ", decrypted_msg)
print("Validation:     ", "SUCCESS" if sample_text == decrypted_msg else "FAILED")


Generated Public Key: (1061, 3233)
Generated Private Key: (941, 3233)

--- RSA Test Results ---
Plain Text:      Research and Instruction Librarian
Encrypted Text:  [90, 2411, 2105, 2411, 2241, 2737, 892, 376, 2175, 2241, 290, 629, 2175, 198, 290, 2105, 1508, 2737, 2375, 892, 1508, 2861, 1087, 290, 2175, 500, 2861, 708, 2737, 2241, 2737, 2861, 2241, 290]
Decrypted Text:  Research and Instruction Librarian
Validation:      SUCCESS


### 2.3 Substitution Cipher
We'll also implement a simple Caesar Substitution cipher for textual data protection, which we will attack in the bonus section.

In [310]:
def substitution_encrypt(text, shift=3):
    result = ""
    for char in text:
        if char.isalpha():
            ascii_offset = 65 if char.isupper() else 97
            result += chr((ord(char) - ascii_offset + shift) % 26 + ascii_offset)
        else:
            result += char
    return result

sample_cipher = substitution_encrypt(sample_text)
print(f"Original Text: {sample_text}")
print(f"Substitution Encrypted Text: {sample_cipher}")


Original Text: Research and Instruction Librarian
Substitution Encrypted Text: Uhvhdufk dqg Lqvwuxfwlrq Oleuduldq


## Step 3: Access Control and Privacy Policies

### Attribute-Based Access Control (ABAC)
We define policies where only authorized users with specific attributes (like Department or Role) can access sensitive columns (like `annual_salary`).

In [311]:
class User:
    def __init__(self, name, role, department):
        self.name = name
        self.role = role
        self.department = department

class ABACPolicy:
    def __init__(self):
        self.policies = {
            "view_salary": lambda user: user.role == "Admin" or user.department == "HR",
            "view_all": lambda user: user.role == "Admin"
        }
        
    def evaluate(self, user, action):
        if action in self.policies:
            return self.policies[action](user)
        return False

abac = ABACPolicy()
print("ABAC Policy Engine initialized.")


ABAC Policy Engine initialized.


In [ ]:
user_hr = User("Alice", "Employee", "HR")
user_analyst = User("Bob", "Employee", "IT")
user_admin = User("Charlie", "Admin", "IT")

print("--- Testing Access Policies ---")
print(f"User {user_hr.name} (Role: {user_hr.role}, Dept: {user_hr.department}) request 'view_salary':", abac.evaluate(user_hr, 'view_salary'))
print(f"User {user_analyst.name} (Role: {user_analyst.role}, Dept: {user_analyst.department}) request 'view_salary':", abac.evaluate(user_analyst, 'view_salary'))
print(f"User {user_admin.name} (Role: {user_admin.role}, Dept: {user_admin.department}) request 'view_all':", abac.evaluate(user_admin, 'view_all'))


--- Testing Access Policies ---
User Alice (Role: Employee, Dept: HR) request 'view_salary': True
User Bob (Role: Employee, Dept: IT) request 'view_salary': False
User Charlie (Role: Admin, Dept: IT) request 'view_all': True


In [ ]:
def fetch_salary_data(user, df):
    if abac.evaluate(user, 'view_salary'):
        return df[['job_title', 'annual_salary']].head(3)
    else:
        return "ACCESS DENIED: Insufficient permissions."

print(f"\n--- Access Attempt by {user_hr.name} ---")
display(fetch_salary_data(user_hr, df))

print(f"\n--- Access Attempt by {user_analyst.name} ---")
print(fetch_salary_data(user_analyst, df))



--- Access Attempt by Alice ---


,job_title,annual_salary
0,Research and Instruction Librarian,55000.0
1,Change & Internal Communications Manager,54600.0
2,Marketing Specialist,34000.0



--- Access Attempt by Bob ---
ACCESS DENIED: Insufficient permissions.


## Step 4: Using Python for Regulatory Compliance

### What is Regulatory Compliance?
Regulatory Compliance refers to the process of ensuring that an organization follows the laws, regulations, guidelines, and specifications relevant to its business. In data governance, this means handling personal and sensitive data according to legal frameworks to protect individual privacy, maintain data integrity, and avoid legal penalties.

### What is the GDPR?
The **General Data Protection Regulation (GDPR)** is a comprehensive data protection law enacted by the European Union (EU) in May 2018. It applies to any organization that processes the personal data of EU residents, regardless of where the organization is located.

- **Key Principles**: Lawfulness, fairness, transparency, purpose limitation, data minimization, accuracy, storage limitation, integrity and confidentiality, and accountability.
- **Core Rights**: Right to access, right to rectification, right to erasure ("right to be forgotten"), right to restrict processing, right to data portability, right to object, and rights related to automated decision-making.
- **Scope**: Applies to all EU member states and extraterritorially to any entity handling EU residents' data.
- **Penalties**: Up to 20 million EUR or 4% of global annual turnover (whichever is higher) for severe infringements.
- **Breach Notification**: Organizations must report data breaches to supervisory authorities within 72 hours.

### What is the CCPA?
The **California Consumer Privacy Act (CCPA)**, effective January 2020, is a state-level statute intended to enhance privacy rights and consumer protection for residents of California, USA.

- **Key Principles**: Consumer right to know, right to delete, right to opt-out, right to non-discrimination, and disclosure obligations.
- **Core Rights**: Right to know what personal information is collected, right to delete personal information, right to opt-out of the sale of personal information, and right to equal service and price.
- **Scope**: Applies to for-profit businesses that collect California residents' personal information and meet certain thresholds (revenue, data volume).
- **Penalties**: Up to $7,500 per intentional violation and $2,500 per unintentional violation; statutory damages for data breaches range from $100 to $750 per consumer per incident.
- **Breach Notification**: No specific CCPA breach notification timeline, but California law generally requires notification without unreasonable delay.

### What is HIPAA?
The **Health Insurance Portability and Accountability Act (HIPAA)**, enacted in 1996 in the United States, sets national standards for the protection of sensitive patient health information.

- **Key Principles**: Privacy Rule ( protects individually identifiable health information), Security Rule (safeguards electronic PHI), Breach Notification Rule, and Minimum Necessary Rule.
- **Core Rights**: Right to access PHI, right to request amendments, right to an accounting of disclosures, right to request restrictions, and right to confidential communications.
- **Scope**: Applies to covered entities (healthcare providers, health plans, healthcare clearinghouses) and their business associates in the United States.
- **Penalties**: Civil monetary penalties range from $137 to $68,928 per violation (adjusted annually); criminal penalties can result in fines up to $250,000 and imprisonment up to 10 years.
- **Breach Notification**: Covered entities must notify affected individuals within 60 days, the Secretary of HHS, and in cases of large breaches, the media.


### Regulatory Compliance Comparison: GDPR vs. CCPA vs. HIPAA

| Feature | GDPR | CCPA | HIPAA |
|---------|------|------|-------|
| **Jurisdiction** | European Union | California, USA | United States (Federal) |
| **Scope** | All EU residents' personal data | California consumers' personal info | Protected Health Information (PHI) of US individuals |
| **Consent Model** | Opt-in (explicit consent required) | Opt-out (can opt-out of sale) | Not primarily consent-based; governed by uses and disclosures |
| **Right to Access** | Yes – comprehensive access to all personal data | Yes – right to know categories and specific pieces | Yes – right to inspect and copy PHI |
| **Right to Delete** | Yes – right to erasure/"right to be forgotten" | Yes – right to delete personal information | Limited – not a general deletion right |
| **Right to Opt-Out** | Yes – of processing for direct marketing | Yes – of sale of personal information | N/A |
| **Data Portability** | Yes – right to receive data in structured format | Yes – via access right | Yes – right to receive copy of PHI |
| **Penalties** | Up to 4% global turnover or 20M EUR | Up to $7,500 per violation; statutory damages $100-$750 per consumer | Civil: $137-$68,928 per violation; Criminal: up to $250,000 + 10 years prison |
| **Breach Notification** | Within 72 hours to authorities | No specific CCPA timeline (general CA law applies) | Within 60 days to individuals; HHS and media for large breaches |
| **Applicability** | Any organization processing EU resident data | For-profit businesses meeting CA thresholds | Covered entities and business associates handling PHI |
| **Key Focus** | Comprehensive privacy rights for individuals | Consumer privacy and transparency | Security and privacy of health information |


In [ ]:
import pandas as pd
import hashlib


def build_ccpa_demo(source_df):
    ccpa = source_df.copy().reset_index(drop=True)

    age_map = {
        'under 18': 16,
        '18-24': 21,
        '25-34': 29,
        '35-44': 39,
        '45-54': 49,
        '55-64': 59,
        '65 or over': 67,
    }

    salary_bins = [-float('inf'), 50000, 75000, 100000, 150000, float('inf')]
    salary_labels = ['<50k', '50k-75k', '75k-100k', '100k-150k', '150k+']

    ccpa['consumer_id'] = [f'C{i + 1:05d}' for i in range(len(ccpa))]
    ccpa['name'] = [f'Respondent {i + 1}' for i in range(len(ccpa))]
    ccpa['email'] = [f'respondent{i + 1}@survey.example' for i in range(len(ccpa))]
    ccpa['phone'] = [f'+1-555-{1000 + (i % 9000):04d}' for i in range(len(ccpa))]
    ccpa['age'] = ccpa['age_group'].astype(str).str.lower().str.strip().map(age_map)
    ccpa['purchase_history'] = (
        ccpa['industry'].fillna('Unknown industry').astype(str).str.strip()
        + ' / '
        + ccpa['job_title'].fillna('Unknown respondent').astype(str).str.strip()
    )
    ccpa['income_bracket'] = pd.cut(
        pd.to_numeric(ccpa['annual_salary'], errors='coerce'),
        bins=salary_bins,
        labels=salary_labels,
    ).astype(object).fillna('unknown')
    ccpa['do_not_sell'] = ccpa.index % 2 == 1

    return ccpa

ccpa_df = build_ccpa_demo(df)

personal_info_cols = ['name', 'email', 'phone', 'age', 'purchase_history', 'income_bracket']
print('=== CCPA Right to Know ===')
print('Personal information categories held:')
for col in personal_info_cols:
    non_null = ccpa_df[col].notna().sum()
    unique = ccpa_df[col].nunique()
    print(f"  {col}: {non_null} records, {unique} unique values")

print()
print('Survey-backed example rows:')
display(ccpa_df[personal_info_cols].head())


=== CCPA Right to Know ===
Personal information categories held:
  name: 5 records, 5 unique values
  email: 5 records, 5 unique values
  phone: 5 records, 5 unique values
  age: 5 records, 5 unique values
  purchase_history: 5 records, 5 unique values
  income_bracket: 5 records, 3 unique values

Sample of personal information:


,name,email,phone,age,purchase_history,income_bracket
0,Alice Smith,alice@example.com,+1-555-0101,34,"Laptop,Mouse",50k-75k
1,Bob Jones,bob@example.com,+1-555-0102,45,"Phone,Case",75k-100k
2,Carol White,carol@example.com,+1-555-0103,29,"Tablet,Keyboard",50k-75k
3,David Lee,david@example.com,+1-555-0104,52,"Monitor,Desk",100k+
4,Eva Brown,eva@example.com,+1-555-0105,38,"Chair,Lamp",75k-100k


In [ ]:
def ccpa_delete(row, cols_to_delete=None):
    if cols_to_delete is None:
        cols_to_delete = ['name', 'email', 'phone', 'purchase_history', 'income_bracket']
    new_row = row.copy()
    for col in cols_to_delete:
        if col in new_row.index:
            new_row[col] = '[DELETED-CCPA]'
    return new_row

target_id = 'C002'
before = ccpa_df[ccpa_df['consumer_id'] == target_id].iloc[0]

ccpa_df_deleted = ccpa_df.copy()
idx = ccpa_df_deleted[ccpa_df_deleted['consumer_id'] == target_id].index[0]
ccpa_df_deleted.loc[idx] = ccpa_delete(ccpa_df_deleted.loc[idx])

after = ccpa_df_deleted[ccpa_df_deleted['consumer_id'] == target_id].iloc[0]

print(f"=== CCPA Right to Delete (Consumer {target_id}) ===")
print('BEFORE:')
print(before[['name', 'email', 'phone', 'purchase_history', 'income_bracket']].to_string())
print()
print('AFTER:')
print(after[['name', 'email', 'phone', 'purchase_history', 'income_bracket']].to_string())


=== CCPA Right to Delete (Consumer C002) ===
BEFORE:
name                      Bob Jones
email               bob@example.com
phone                   +1-555-0102
purchase_history         Phone,Case
income_bracket             75k-100k

AFTER:
name                [DELETED-CCPA]
email               [DELETED-CCPA]
phone               [DELETED-CCPA]
purchase_history    [DELETED-CCPA]
income_bracket      [DELETED-CCPA]


In [ ]:
def apply_do_not_sell(df, sellable_col='purchase_history'):
    df = df.copy()
    df['sellable_data'] = df.apply(
        lambda row: row[sellable_col] if not row.get('do_not_sell', False) else '[OPTED-OUT]',
        axis=1
    )
    return df

ccpa_df_opt = apply_do_not_sell(ccpa_df_deleted, sellable_col='purchase_history')

print('=== CCPA Do Not Sell Flag ===')
print(ccpa_df_opt[['consumer_id', 'name', 'do_not_sell', 'purchase_history', 'sellable_data']])

opted_out = ccpa_df_opt[ccpa_df_opt['do_not_sell'] == True]
print(f"\nConsumers opted-out: {len(opted_out)}")
print('Sellable dataset rows (do_not_sell=False):')
sellable = ccpa_df_opt[ccpa_df_opt['do_not_sell'] == False]
print(sellable[['consumer_id', 'sellable_data']].to_string(index=False))


=== CCPA Do Not Sell Flag ===
  consumer_id            name  do_not_sell purchase_history    sellable_data
0        C001     Alice Smith        False     Laptop,Mouse     Laptop,Mouse
1        C002  [DELETED-CCPA]         True   [DELETED-CCPA]      [OPTED-OUT]
2        C003     Carol White        False  Tablet,Keyboard  Tablet,Keyboard
3        C004       David Lee        False     Monitor,Desk     Monitor,Desk
4        C005       Eva Brown         True       Chair,Lamp      [OPTED-OUT]

Consumers opted-out: 2
Sellable dataset rows (do_not_sell=False):
consumer_id   sellable_data
       C001    Laptop,Mouse
       C003 Tablet,Keyboard
       C004    Monitor,Desk


In [ ]:

try:
    from datasketches import hll_sketch
    hll = hll_sketch(12)
    for email in ccpa_df['email']:
        hll.update(str(email))
    estimated_unique = hll.get_estimate()
    print('=== CCPA Data Minimization (datasketches HyperLogLog) ===')
    print(f'Estimated unique emails (cardinality): {estimated_unique:.0f}')
    print(f"Actual unique emails: {ccpa_df['email'].nunique()}")
except ImportError:
    print('datasketches not installed; using native set approximation')
    estimated_unique = len(set(ccpa_df['email']))
    print(f'Unique emails (native set): {estimated_unique}')

def sha256_hash(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

ccpa_anon = ccpa_df_opt.copy()
for col in ['name', 'email', 'phone']:
    ccpa_anon[f'{col}_hash'] = ccpa_anon[col].apply(sha256_hash)

print('\n=== CCPA Anonymization (SHA-256) ===')
display_cols = ['consumer_id', 'name_hash', 'email_hash', 'phone_hash']
print(ccpa_anon[display_cols].to_string(index=False))


=== CCPA Data Minimization (datasketches HyperLogLog) ===
Estimated unique emails (cardinality): 5
Actual unique emails: 5

=== CCPA Anonymization (SHA-256) ===
consumer_id                                                        name_hash                                                       email_hash                                                       phone_hash
       C001 8ae10dfc9a69f97dcc91e1f51bced3aff442bc57a869e04126bd0ca7cb4af29d ff8d9819fc0e12bf0d24892e45987e249a28dce836a85cad60e28eaaa8c6d976 f7c4f54a232defdba706794e0534207b1a761d11ad56c1c161627d9f036170e3
       C002 569a1418e41a03c459cc43719169249b4ffad11980d9daf4d5e0782d9f9e3d95 569a1418e41a03c459cc43719169249b4ffad11980d9daf4d5e0782d9f9e3d95 569a1418e41a03c459cc43719169249b4ffad11980d9daf4d5e0782d9f9e3d95
       C003 2bba1d047eb0226670fc59b93d997ead92f70095b96c1c40ab00ae3b642a3cde e0d47ca1bc1eb62e650fc1fd660a9bfbf7cba8dc6337d81df7ea9aa9071a24a5 dc9c88c972fc262a6f5a653655052efe570e8fc1c3b3695a3a642dd330167e88
       C004

In [ ]:
import pandas as pd
import hashlib
from datetime import datetime


def build_hipaa_demo(source_df):
    hipaa = source_df.copy().reset_index(drop=True)

    age_map = {
        'under 18': 16,
        '18-24': 21,
        '25-34': 29,
        '35-44': 39,
        '45-54': 49,
        '55-64': 59,
        '65 or over': 67,
    }

    ref_dates = pd.to_datetime(hipaa['timestamp'], errors='coerce').fillna(pd.Timestamp('2021-01-01'))
    age_years = hipaa['age_group'].astype(str).str.lower().str.strip().map(age_map).fillna(42).astype(int)
    salary_values = pd.to_numeric(hipaa['annual_salary'], errors='coerce').fillna(0)

    hipaa['patient_id'] = [f'P{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['name'] = [f'Patient {i + 1}' for i in range(len(hipaa))]
    hipaa['ssn'] = [f'{100 + (i % 900):03d}-{10 + (i % 90):02d}-{1000 + (i % 9000):04d}' for i in range(len(hipaa))]
    hipaa['dob'] = (ref_dates - pd.to_timedelta(age_years * 365, unit='D')).dt.strftime('%Y-%m-%d')
    hipaa['email'] = [f'patient{i + 1}@healthmail.com' for i in range(len(hipaa))]
    hipaa['phone'] = [f'+1-555-{2000 + (i % 8000):04d}' for i in range(len(hipaa))]
    hipaa['address'] = (
        hipaa['city'].fillna('Unknown city').astype(str).str.strip()
        + ', '
        + hipaa['us_state'].fillna('Unknown state').astype(str).str.strip()
        + ', '
        + hipaa['country'].fillna('Unknown country').astype(str).str.strip()
    )
    hipaa['medical_record_number'] = [f'MRN-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['health_plan_id'] = [f'HP-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['account_number'] = [f'ACCT-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['certificate_number'] = [f'CERT-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['vehicle_id'] = [f'VEH-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['device_id'] = [f'DEV-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['biometric'] = [f'BIO-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['photo_url'] = [f'https://survey.example/photos/{i + 1}' for i in range(len(hipaa))]
    hipaa['diagnosis'] = (
        'Occupational review: '
        + hipaa['industry'].fillna('unknown').astype(str).str.strip()
    )
    hipaa['treatment'] = (
        'Workplace review for '
        + hipaa['job_title'].fillna('respondent').astype(str).str.strip()
    )
    hipaa['billing_amount'] = (salary_values * 0.01).round(2)
    hipaa['admission_date'] = ref_dates.dt.strftime('%Y-%m-%d')
    hipaa['provider_name'] = [f'Dr. Provider {i % 5 + 1}' for i in range(len(hipaa))]
    hipaa['provider_npi'] = [f'NPI-{i + 1:05d}' for i in range(len(hipaa))]
    hipaa['url'] = [f'https://survey.example/patient/{i + 1}' for i in range(len(hipaa))]
    hipaa['ip_address'] = [f'10.0.{(i // 255) % 255}.{i % 255 + 1}' for i in range(len(hipaa))]
    hipaa['fax'] = [f'+1-555-{3000 + (i % 7000):04d}' for i in range(len(hipaa))]
    hipaa['age_group'] = hipaa['age_group']
    hipaa['age'] = age_years

    return hipaa

hipaa_df = build_hipaa_demo(df)

hipaa_18_identifiers = [
    'name', 'ssn', 'dob', 'email', 'phone', 'address', 'medical_record_number',
    'health_plan_id', 'account_number', 'certificate_number', 'vehicle_id',
    'device_id', 'biometric', 'photo_url', 'provider_name', 'provider_npi',
    'url', 'ip_address', 'fax'
]

print('=== HIPAA PHI Inventory (18 Identifiers) ===')
for ident in hipaa_18_identifiers:
    present = ident in hipaa_df.columns
    status = 'PRESENT' if present else 'MISSING'
    print(f'  {ident}: {status}')

print(f'\nDataset shape: {hipaa_df.shape}')
print('Survey-backed PHI example columns:')
print(hipaa_df[['patient_id', 'name', 'ssn', 'dob', 'email', 'phone']].head().to_string(index=False))


=== HIPAA PHI Inventory (18 Identifiers) ===
  name: PRESENT
  ssn: PRESENT
  dob: PRESENT
  email: PRESENT
  phone: PRESENT
  address: PRESENT
  medical_record_number: PRESENT
  health_plan_id: PRESENT
  account_number: PRESENT
  certificate_number: PRESENT
  vehicle_id: PRESENT
  device_id: PRESENT
  biometric: PRESENT
  photo_url: PRESENT
  provider_name: PRESENT
  provider_npi: PRESENT
  url: PRESENT
  ip_address: PRESENT
  fax: PRESENT

Dataset shape: (5, 24)
Sample PHI columns:
patient_id     name         ssn        dob               email       phone
      P001 John Doe 123-45-6789 1985-03-15 john@healthmail.com +1-555-1001
      P002 Jane Roe 987-65-4321 1990-07-22 jane@healthmail.com +1-555-1002
      P003  Jim Poe 456-78-9123 1978-11-05  jim@healthmail.com +1-555-1003
      P004 Jack Coe 321-54-9876 2000-01-10 jack@healthmail.com +1-555-1004
      P005 Jill Moe 789-12-3456 1995-05-30 jill@healthmail.com +1-555-1005


In [ ]:
def role_based_access(df, role):
    role_permissions = {
        'doctor': ['patient_id', 'name', 'dob', 'diagnosis', 'treatment', 'admission_date', 'provider_name'],
        'nurse': ['patient_id', 'name', 'dob', 'diagnosis', 'admission_date'],
        'billing': ['patient_id', 'name', 'billing_amount', 'account_number', 'health_plan_id'],
        'researcher': ['patient_id', 'diagnosis', 'treatment', 'age_group'],  # de-identified
        'admin': list(df.columns)
    }
    allowed = role_permissions.get(role, [])
    return df[[c for c in allowed if c in df.columns]]

roles = ['doctor', 'nurse', 'billing', 'researcher']
for role in roles:
    subset = role_based_access(hipaa_df, role)
    print(f"=== {role.upper()} view ({len(subset.columns)} columns) ===")
    print(list(subset.columns))
    print(subset.head(2).to_string(index=False))
    print()

def hash_ephi(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

ephi_cols = ['name', 'ssn', 'email', 'phone', 'medical_record_number']
hipaa_hashed = hipaa_df.copy()
for col in ephi_cols:
    hipaa_hashed[f'{col}_hash'] = hipaa_hashed[col].apply(hash_ephi)

print('=== Technical Safeguards: ePHI Hashed ===')
print(hipaa_hashed[['patient_id', 'name_hash', 'ssn_hash', 'email_hash']].head().to_string(index=False))


=== DOCTOR view (7 columns) ===
['patient_id', 'name', 'dob', 'diagnosis', 'treatment', 'admission_date', 'provider_name']
patient_id     name        dob       diagnosis       treatment admission_date provider_name
      P001 John Doe 1985-03-15    Hypertension    Medication A     2024-01-10     Dr. Smith
      P002 Jane Roe 1990-07-22 Diabetes Type 2 Insulin Therapy     2024-02-14     Dr. Jones

=== NURSE view (5 columns) ===
['patient_id', 'name', 'dob', 'diagnosis', 'admission_date']
patient_id     name        dob       diagnosis admission_date
      P001 John Doe 1985-03-15    Hypertension     2024-01-10
      P002 Jane Roe 1990-07-22 Diabetes Type 2     2024-02-14

=== BILLING view (5 columns) ===
['patient_id', 'name', 'billing_amount', 'account_number', 'health_plan_id']
patient_id     name  billing_amount account_number health_plan_id
      P001 John Doe           150.0       ACCT-101         HP-111
      P002 Jane Roe           320.0       ACCT-102         HP-222

=== RESEARCH

In [ ]:
audit_log = []

def log_access(user, role, action, patient_id, success=True):
    audit_log.append({
        'timestamp': datetime.now().isoformat(),
        'user': user,
        'role': role,
        'action': action,
        'patient_id': patient_id,
        'success': success
    })

log_access('dr_smith', 'doctor', 'view_diagnosis', 'P001')
log_access('nurse_amy', 'nurse', 'view_admission', 'P002')
log_access('billing_joe', 'billing', 'view_billing_amount', 'P003')
log_access('researcher_kim', 'researcher', 'export_diagnosis', 'P004')
log_access('dr_smith', 'doctor', 'edit_treatment', 'P001')

audit_df = pd.DataFrame(audit_log)
print('=== HIPAA Audit Log ===')
print(audit_df.to_string(index=False))

def compute_record_hash(row, cols):
    record_str = '|'.join(str(row.get(c, '')) for c in cols)
    return hashlib.sha256(record_str.encode()).hexdigest()

integrity_cols = ['patient_id', 'name', 'dob', 'diagnosis', 'treatment']
hipaa_df['record_hash'] = hipaa_df.apply(lambda r: compute_record_hash(r, integrity_cols), axis=1)

print('\n=== Data Integrity: Record-Level Hashes ===')
print(hipaa_df[['patient_id', 'record_hash']].to_string(index=False))

test_idx = 0
recomputed = compute_record_hash(hipaa_df.iloc[test_idx], integrity_cols)
stored = hipaa_df.iloc[test_idx]['record_hash']
print(f"\nIntegrity check for {hipaa_df.iloc[test_idx]['patient_id']}: {'PASS' if recomputed == stored else 'FAIL'}")


=== HIPAA Audit Log ===
                 timestamp           user       role              action patient_id  success
2026-05-09T13:17:00.912633       dr_smith     doctor      view_diagnosis       P001     True
2026-05-09T13:17:00.912686      nurse_amy      nurse      view_admission       P002     True
2026-05-09T13:17:00.912714    billing_joe    billing view_billing_amount       P003     True
2026-05-09T13:17:00.912740 researcher_kim researcher    export_diagnosis       P004     True
2026-05-09T13:17:00.912763       dr_smith     doctor      edit_treatment       P001     True

=== Data Integrity: Record-Level Hashes ===
patient_id                                                      record_hash
      P001 1ea68113294528014e62b50f1b6013a5992632ba2ae730767ab8d0042c01a5f8
      P002 e3972019e5c780f131ca9c0c23ccb12121e9d47d0c1fed0a8901e5deedbe35ef
      P003 0fb703faf4d6c425ae50cfedf7f3b632f6a413eb3da801ee1dff53fca66e329c
      P004 6fdb0104cb84222e1dcd1e39fc827780b3ecf2798fd3a01c7b5ed893b2

In [ ]:
def safe_harbor_deidentify(df):
    df_deid = df.copy()
    drop_cols = ['name', 'ssn', 'email', 'phone', 'address', 'medical_record_number',
                 'health_plan_id', 'account_number', 'certificate_number',
                 'vehicle_id', 'device_id', 'biometric', 'photo_url',
                 'provider_name', 'provider_npi', 'url', 'ip_address', 'fax']
    df_deid = df_deid.drop(columns=[c for c in drop_cols if c in df_deid.columns])
    df_deid['dob_year'] = pd.to_datetime(df_deid['dob'], errors='coerce').dt.year
    df_deid = df_deid.drop(columns=['dob'], errors='ignore')
    df_deid['admission_year'] = pd.to_datetime(df_deid['admission_date'], errors='coerce').dt.year
    df_deid = df_deid.drop(columns=['admission_date'], errors='ignore')
    df_deid = df_deid.drop(columns=['age'], errors='ignore')
    return df_deid

hipaa_deid = safe_harbor_deidentify(hipaa_df)
print('=== Safe Harbor De-identification ===')
print(f'Original columns: {list(hipaa_df.columns)}')
print(f'De-identified columns: {list(hipaa_deid.columns)}')
print(hipaa_deid.head().to_string(index=False))

compliance_findings = []

# Check 1: Are direct identifiers removed?
remaining_identifiers = [c for c in hipaa_18_identifiers if c in hipaa_deid.columns]
compliance_findings.append({
    'check': 'Direct identifiers removed',
    'status': 'PASS' if len(remaining_identifiers) == 0 else 'FAIL',
    'details': f'{len(remaining_identifiers)} identifiers remaining: {remaining_identifiers}' if remaining_identifiers else 'All 18 identifiers removed or generalized'
})

# Check 2: Are dates generalized to year?
has_year_only = 'dob_year' in hipaa_deid.columns and 'dob' not in hipaa_deid.columns
compliance_findings.append({
    'check': 'Dates generalized to year',
    'status': 'PASS' if has_year_only else 'FAIL',
    'details': 'Dates truncated to year only' if has_year_only else 'Full dates still present'
})

# Check 3: Are record hashes present for integrity?
has_hashes = 'record_hash' in hipaa_df.columns
compliance_findings.append({
    'check': 'Record-level integrity hashes present',
    'status': 'PASS' if has_hashes else 'FAIL',
    'details': 'SHA-256 record hashes computed' if has_hashes else 'No integrity hashes found'
})

findings_df = pd.DataFrame(compliance_findings)
print('\n=== Security Audit Findings ===')
print(findings_df.to_string(index=False))


=== Safe Harbor De-identification ===
Original columns: ['patient_id', 'name', 'ssn', 'dob', 'email', 'phone', 'address', 'medical_record_number', 'health_plan_id', 'account_number', 'certificate_number', 'vehicle_id', 'device_id', 'biometric', 'photo_url', 'diagnosis', 'treatment', 'billing_amount', 'admission_date', 'provider_name', 'provider_npi', 'url', 'ip_address', 'fax', 'record_hash']
De-identified columns: ['patient_id', 'diagnosis', 'treatment', 'billing_amount', 'record_hash', 'dob_year', 'admission_year']
patient_id       diagnosis       treatment  billing_amount                                                      record_hash  dob_year  admission_year
      P001    Hypertension    Medication A           150.0 1ea68113294528014e62b50f1b6013a5992632ba2ae730767ab8d0042c01a5f8      1985            2024
      P002 Diabetes Type 2 Insulin Therapy           320.0 e3972019e5c780f131ca9c0c23ccb12121e9d47d0c1fed0a8901e5deedbe35ef      1990            2024
      P003          Asthma 

In [ ]:
def mask_name(name):
    parts = str(name).split()
    return ' '.join([p[0] + '*' * (len(p) - 1) for p in parts])

def mask_email(email):
    if '@' not in str(email):
        return email
    local, domain = str(email).split('@')
    masked_local = local[0] + '*' * (len(local) - 1)
    return f'{masked_local}@{domain}'

def mask_phone(phone):
    digits = ''.join(ch for ch in str(phone) if ch.isdigit())
    if len(digits) >= 4:
        return 'X' * (len(digits) - 4) + digits[-4:]
    return 'X' * len(digits)

hipaa_masked = hipaa_df.copy()
hipaa_masked['name_masked'] = hipaa_masked['name'].apply(mask_name)
hipaa_masked['email_masked'] = hipaa_masked['email'].apply(mask_email)
hipaa_masked['phone_masked'] = hipaa_masked['phone'].apply(mask_phone)

print('=== HIPAA Data Masking (Hippo-style) ===')
mask_cols = ['patient_id', 'name', 'name_masked', 'email', 'email_masked', 'phone', 'phone_masked']
print(hipaa_masked[mask_cols].to_string(index=False))

print('\nMasking summary:')
print(f'  Names masked: {hipaa_masked["name"].notna().sum()}')
print(f'  Emails masked: {hipaa_masked["email"].notna().sum()}')
print(f'  Phones masked: {hipaa_masked["phone"].notna().sum()}')


=== HIPAA Data Masking (Hippo-style) ===
patient_id     name name_masked               email        email_masked       phone phone_masked
      P001 John Doe    J*** D** john@healthmail.com j***@healthmail.com +1-555-1001     XXXX1001
      P002 Jane Roe    J*** R** jane@healthmail.com j***@healthmail.com +1-555-1002     XXXX1002
      P003  Jim Poe     J** P**  jim@healthmail.com  j**@healthmail.com +1-555-1003     XXXX1003
      P004 Jack Coe    J*** C** jack@healthmail.com j***@healthmail.com +1-555-1004     XXXX1004
      P005 Jill Moe    J*** M** jill@healthmail.com j***@healthmail.com +1-555-1005     XXXX1005

Masking summary:
  Names masked: 5
  Emails masked: 5
  Phones masked: 5


## Step 5: Bonus 1 - Attacks

### 5.1 Attacking the Substitution Cipher
We perform a brute-force frequency attack to guess the shift.

In [ ]:
def substitution_encrypt(text, shift):
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    translated = []
    for ch in str(text):
        lower = ch.lower()
        if lower in alphabet:
            idx = (alphabet.index(lower) + shift) % 26
            new_ch = alphabet[idx]
            translated.append(new_ch.upper() if ch.isupper() else new_ch)
        else:
            translated.append(ch)
    return ''.join(translated)

def substitution_decrypt(ciphertext, shift):
    return substitution_encrypt(ciphertext, -shift)

attack_row = df[['job_title', 'industry', 'country']].dropna(subset=['job_title', 'industry']).iloc[0]
attack_text = f"{attack_row['job_title']} | {attack_row['industry']} | {attack_row['country']}"
attack_shift = 7
sample_text = attack_text
sample_cipher = substitution_encrypt(sample_text, attack_shift)

print("Dataset row used for attack:")
print(sample_text)
print("Target Ciphertext:", sample_cipher)
print("\nStarting brute-force attack...")
for shift_guess in range(1, 26):
    guess = substitution_decrypt(sample_cipher, shift_guess)
    if guess == sample_text:
        print(f"--> SUCCESS! Found matching text with shift {shift_guess}: {guess}")
        break


Target Ciphertext: Uhvhdufk dqg Lqvwuxfwlrq Oleuduldq

Starting brute-force attack...
--> SUCCESS! Found matching text with shift 3: Research and Instruction Librarian


### 5.2 Attacking the RSA Algorithm
We attack our small-prime RSA implementation by factoring the public modulus $N$.

In [324]:
n_target = public[1]
e_target = public[0]

print(f"Target Public Modulus (N) to factorize: {n_target}")

def factorize_n(n):
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return i, n // i
    return None, None

p_cracked, q_cracked = factorize_n(n_target)
print(f"Factoring complete. Discovered p={p_cracked}, q={q_cracked}")

if p_cracked:
    phi_cracked = (p_cracked - 1) * (q_cracked - 1)
    d_cracked = pow(e_target, -1, phi_cracked)
    hacked_private_key = (d_cracked, n_target)
    
    print("\nHacked Private Key:", hacked_private_key)
    print("Original Private Key:", private)
    
    hacked_msg = rsa_decrypt(hacked_private_key, encrypted_msg)
    print("\nDecrypted message using HACKED key:", hacked_msg)


Target Public Modulus (N) to factorize: 3233
Factoring complete. Discovered p=53, q=61

Hacked Private Key: (941, 3233)
Original Private Key: (941, 3233)

Decrypted message using HACKED key: Research and Instruction Librarian


## Step 6: Bonus 2 - Machine Learning Preprocessing

We will preprocess the original data to prepare it for predicting `annual_salary`.

In [ ]:
features = ['age_group', 'industry', 'years_exp_overall_num']
target = 'annual_salary'

ml_df = df[features + [target]].dropna().copy()
print(f"Rows after dropping NA values: {ml_df.shape[0]}")

print("\nSample data BEFORE Label Encoding and Scaling:")
display(ml_df.head(3))


Rows after dropping NA values: 27978

Sample data BEFORE Label Encoding and Scaling:


,age_group,industry,years_exp_overall_num,annual_salary
0,25-34,Education (Higher Education),6.0,55000.0
1,25-34,Computing or Tech,9.0,54600.0
2,25-34,"Accounting, Banking & Finance",3.0,34000.0


In [ ]:
from sklearn.preprocessing import LabelEncoder 
from sklearn.preprocessing import StandardScaler  
le = LabelEncoder()
ml_df['age_group_encoded'] = le.fit_transform(ml_df['age_group'])
ml_df['industry_encoded'] = le.fit_transform(ml_df['industry'])

X = ml_df[['age_group_encoded', 'industry_encoded', 'years_exp_overall_num']]
y = ml_df[target]

print("Data AFTER Label Encoding:")
display(X.head(3))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nData AFTER Standard Scaling (first 3 rows):")
print(X_scaled[:3])


Data AFTER Label Encoding:


,age_group_encoded,industry_encoded,years_exp_overall_num
0,1,287,6.0
1,1,195,9.0
2,1,18,3.0



Data AFTER Standard Scaling (first 3 rows):
[[-0.73131068 -0.51058945 -0.85395398]
 [-0.73131068 -0.87798927 -0.48680202]
 [-0.73131068 -1.58483457 -1.22110595]]


In [ ]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Preprocessing Complete!")
print("Training Data Shape (X, y):", X_train.shape, y_train.shape)
print("Testing Data Shape  (X, y):", X_test.shape, y_test.shape)


Preprocessing Complete!
Training Data Shape (X, y): (22382, 3) (22382,)
Testing Data Shape  (X, y): (5596, 3) (5596,)


## Bonus A: Data Governance Maturity Scorecard

An interactive governance health assessment that scores the dataset across five dimensions: Completeness, Validity, Consistency, Uniqueness, and Accuracy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

completeness = (1 - df.isnull().mean().mean()) * 100

valid_rows = len(df) - 7  
validity = (valid_rows / len(df)) * 100

consistent_count = 0
consistent_count += (df['country'] == 'United States').sum()  
consistent_count += df['currency_clean'].isin(['USD','GBP','CAD','AUD','EUR']).sum()
consistent_count += df['age_group'].isin(VALID_AGE_GROUPS).sum()
consistent_count += df['years_exp_overall'].isin(VALID_EXP).sum()
consistency = (consistent_count / (len(df) * 4)) * 100

uniqueness = ((len(df) - df.duplicated().sum()) / len(df)) * 100

salary_in_range = ((df['annual_salary'] >= 1_000) & (df['annual_salary'] <= 5_000_000)).sum()
accuracy = (salary_in_range / len(df)) * 100

scores = {
    'Completeness': round(completeness, 1),
    'Validity': round(validity, 1),
    'Consistency': round(min(consistency, 100), 1),
    'Uniqueness': round(uniqueness, 1),
    'Accuracy': round(accuracy, 1),
}

categories = list(scores.keys())
values = list(scores.values())
values += values[:1]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.fill(angles, values, color='#4C72B0', alpha=0.25)
ax.plot(angles, values, color='#4C72B0', linewidth=2, marker='o')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title('Data Governance Maturity Scorecard', fontsize=14, fontweight='bold', pad=20)
for angle, val, cat in zip(angles[:-1], values[:-1], categories):
    ax.annotate(f'{val}%', xy=(angle, val), xytext=(angle, val + 6),
                ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('governance_scorecard.png', dpi=150)
plt.show()

overall = round(np.mean(list(scores.values())), 1)
print(f'Overall Governance Score: {overall}/100')
for k, v in scores.items():
    print(f'  {k}: {v}%')


Overall Governance Score: 97.7/100
  Completeness: 93.2%
  Validity: 100.0%
  Consistency: 95.3%
  Uniqueness: 100.0%
  Accuracy: 100.0%


C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\3646731929.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Bonus B: Differential Privacy Demonstration

We apply the Laplace mechanism to salary statistics to demonstrate privacy-preserving analytics. By injecting calibrated noise, we protect individual respondent salaries while preserving aggregate accuracy.


In [ ]:
import numpy as np

def laplace_noise(scale, size=1):
    return np.random.laplace(0, scale, size)

epsilon_values = [0.1, 0.5, 1.0, 5.0]
salary_series = pd.to_numeric(df['annual_salary'], errors='coerce').dropna()
if salary_series.empty:
    raise ValueError('annual_salary has no usable values for the differential privacy demo.')

sensitivity_mean = (salary_series.max() - salary_series.min()) / len(salary_series)
sensitivity_median = sensitivity_mean

print('=== Differential Privacy: Salary Statistics (Laplace Mechanism) ===')
print(f"True Mean:   ${salary_series.mean():,.0f}")
print(f"True Median: ${salary_series.median():,.0f}")
print()

results = []
for eps in epsilon_values:
    scale_mean = sensitivity_mean / eps
    scale_median = sensitivity_median / eps
    noisy_mean = salary_series.mean() + laplace_noise(scale_mean)[0]
    noisy_median = salary_series.median() + laplace_noise(scale_median)[0]
    results.append({
        'epsilon': eps,
        'noisy_mean': round(noisy_mean, 0),
        'mean_error': round(abs(noisy_mean - salary_series.mean()), 0),
        'noisy_median': round(noisy_median, 0),
        'median_error': round(abs(noisy_median - salary_series.median()), 0)
    })
    print(f"epsilon={eps}: Noisy Mean=${noisy_mean:,.0f} (error=${abs(noisy_mean - salary_series.mean()):,.0f}) | Noisy Median=${noisy_median:,.0f}")

dp_df = pd.DataFrame(results)
print()
print(dp_df.to_string(index=False))


=== Differential Privacy: Salary Statistics (Laplace Mechanism) ===
True Mean:   $92,355
True Median: $78,500

epsilon=0.1: Noisy Mean=$92,186 (error=$169) | Noisy Median=$77,097
epsilon=0.5: Noisy Mean=$91,724 (error=$631) | Noisy Median=$78,443
epsilon=1.0: Noisy Mean=$92,290 (error=$66) | Noisy Median=$78,516
epsilon=5.0: Noisy Mean=$92,287 (error=$68) | Noisy Median=$78,481

 epsilon  noisy_mean  mean_error  noisy_median  median_error
     0.1     92186.0       169.0       77097.0        1403.0
     0.5     91724.0       631.0       78443.0          57.0
     1.0     92290.0        66.0       78516.0          16.0
     5.0     92287.0        68.0       78481.0          19.0


## Bonus C: Automated PII Risk Scanner

A lightweight governance tool that scans every text column in the dataset for patterns resembling personally identifiable information (PII) and produces a risk heatmap. This simulates enterprise data-discovery capabilities.


In [ ]:
import re
import seaborn as sns
import matplotlib.pyplot as plt

pii_patterns = {
    'email': re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'),
    'phone': re.compile(r'(\+1[-\s]?)?\(?[0-9]{3}\)?[-\s]?[0-9]{3}[-\s]?[0-9]{4}'),
    'ssn': re.compile(r'\b\d{3}-\d{2}-\d{4}\b'),
    'ip_address': re.compile(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'),
    'url': re.compile(r'https?://[^\s]+'),
    'credit_card': re.compile(r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
    'zip_plus4': re.compile(r'\b\d{5}-\d{4}\b'),
}

text_cols = df.select_dtypes(include=['object']).columns.tolist()

risk_matrix = pd.DataFrame(0, index=text_cols, columns=list(pii_patterns.keys()))

for col in text_cols:
    for pii_name, pattern in pii_patterns.items():
        matches = df[col].dropna().astype(str).apply(lambda x: bool(pattern.search(x))).sum()
        risk_score = min(matches / len(df) * 100, 100)
        risk_matrix.loc[col, pii_name] = risk_score

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(risk_matrix, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax, cbar_kws={'label': 'Risk Score (%)'})
ax.set_title('PII Risk Heatmap by Column', fontsize=14, fontweight='bold')
ax.set_xlabel('PII Pattern', fontsize=11)
ax.set_ylabel('Dataset Column', fontsize=11)
plt.tight_layout()
plt.savefig('pii_risk_heatmap.png', dpi=150)
plt.show()

max_per_col = risk_matrix.max(axis=1).sort_values(ascending=False)
print('=== Top 10 Columns by PII Risk ===')
for col, score in max_per_col.head(10).items():
    if score > 0:
        top_pattern = risk_matrix.loc[col].idxmax()
        print(f"  {col}: {score:.1f}% risk ({top_pattern})")


C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\1488153936.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.007130124777183601' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  risk_matrix.loc[col, pii_name] = risk_score
C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\1488153936.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '18.238859180035654' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  risk_matrix.loc[col, pii_name] = risk_score


=== Top 10 Columns by PII Risk ===
  phone_number: 100.0% risk (phone)
  hashed_password: 18.2% risk (phone)
  hashed_phone_number: 18.1% risk (phone)
  job_title_context: 0.0% risk (url)
  income_context: 0.0% risk (url)


C:\Users\Omar Darwish\AppData\Local\Temp\ipykernel_14452\1488153936.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

fx_to_usd = {
    'USD': 1.0,
    'GBP': 1.37,
    'CAD': 0.80,
    'AUD': 0.74,
    'EUR': 1.18,
    'CHF': 1.09,
    'SEK': 0.12,
    'NOK': 0.11,
    'DKK': 0.16,
    'INR': 0.013,
    'NZD': 0.70,
    'HKD': 0.13,
    'SGD': 0.74,
    'ZAR': 0.067,
    'JPY': 0.0091,
    'MXN': 0.050,
    'BRL': 0.19,
}

salary_frame = df.copy()
salary_frame['salary_raw_num'] = pd.to_numeric(salary_frame['annual_salary'], errors='coerce')
salary_frame['currency_for_normalization'] = salary_frame.get('currency_clean', salary_frame.get('currency', 'USD')).fillna('USD').astype(str).str.upper()
salary_frame['fx_to_usd'] = salary_frame['currency_for_normalization'].map(fx_to_usd).fillna(1.0)
salary_frame['salary_usd'] = salary_frame['salary_raw_num'] / salary_frame['fx_to_usd']

usable_salary = salary_frame['salary_usd'].dropna()
if usable_salary.empty:
    raise ValueError('No usable salary values found for normalization.')

lower_clip = usable_salary.quantile(0.01)
upper_clip = usable_salary.quantile(0.99)
salary_frame['salary_normalized'] = salary_frame['salary_usd'].clip(lower=lower_clip, upper=upper_clip)

summary_stats = pd.DataFrame({
    'metric': ['count', 'mean', 'median', 'std', 'p01', 'p99'],
    'before_cleaning': [
        salary_frame['salary_raw_num'].count(),
        salary_frame['salary_raw_num'].mean(),
        salary_frame['salary_raw_num'].median(),
        salary_frame['salary_raw_num'].std(),
        salary_frame['salary_raw_num'].quantile(0.01),
        salary_frame['salary_raw_num'].quantile(0.99),
    ],
    'after_normalization': [
        salary_frame['salary_normalized'].count(),
        salary_frame['salary_normalized'].mean(),
        salary_frame['salary_normalized'].median(),
        salary_frame['salary_normalized'].std(),
        salary_frame['salary_normalized'].quantile(0.01),
        salary_frame['salary_normalized'].quantile(0.99),
    ],
})

print('=== What-if Salary Normalization ===')
print('Currency conversion to USD-equivalent plus 1st/99th percentile clipping.')
print(summary_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(salary_frame['salary_raw_num'].dropna(), bins=40, ax=axes[0], color='#8C510A')
axes[0].set_title('Before normalization')
axes[0].set_xlabel('Annual salary')
axes[0].set_ylabel('Count')
sns.histplot(salary_frame['salary_normalized'].dropna(), bins=40, ax=axes[1], color='#01665E')
axes[1].set_title('After normalization')
axes[1].set_xlabel('USD-equivalent salary')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=salary_frame['salary_raw_num'].dropna(), ax=axes[0], color='#D8B365')
axes[0].set_title('Raw salary distribution')
axes[0].set_xlabel('Annual salary')
sns.boxplot(x=salary_frame['salary_normalized'].dropna(), ax=axes[1], color='#5AB4AC')
axes[1].set_title('Normalized salary distribution')
axes[1].set_xlabel('USD-equivalent salary')
plt.tight_layout()
plt.show()


In [ ]:
print('\n=== Representational Imbalance Analysis ===')

bias_dims = ['gender', 'education', 'country']
bias_tables = []

for dim in bias_dims:
    if dim not in salary_frame.columns:
        continue
    series = salary_frame[dim].astype(str).replace({'nan': np.nan}).dropna()
    if series.empty:
        continue
    top_categories = series.value_counts().head(8)
    dim_total = int(top_categories.sum())
    dim_groups = len(top_categories)
    expected_share = 1 / dim_groups if dim_groups else 0
    for category, count in top_categories.items():
        share = count / dim_total
        bias_index = share / expected_share if expected_share else np.nan
        bias_tables.append({
            'dimension': dim,
            'category': category,
            'count': int(count),
            'share_pct': round(share * 100, 1),
            'bias_index': round(bias_index, 2),
            'flag': 'OVER' if bias_index > 1.2 else ('UNDER' if bias_index < 0.8 else 'BALANCED')
        })

bias_df = pd.DataFrame(bias_tables)
if not bias_df.empty:
    display(bias_df.sort_values(['dimension', 'share_pct'], ascending=[True, False]).reset_index(drop=True))

    fig, axes = plt.subplots(len(bias_dims), 1, figsize=(12, 4 * len(bias_dims)))
    if len(bias_dims) == 1:
        axes = [axes]
    for ax, dim in zip(axes, bias_dims):
        subset = bias_df[bias_df['dimension'] == dim].sort_values('share_pct', ascending=True)
        if subset.empty:
            ax.axis('off')
            continue
        colors = subset['flag'].map({'OVER': '#D73027', 'UNDER': '#4575B4', 'BALANCED': '#91BFDB'})
        ax.barh(subset['category'], subset['share_pct'], color=colors)
        ax.set_title(f'{dim.title()} representation share among top categories')
        ax.set_xlabel('Share (%)')
        ax.set_ylabel(dim.title())
    plt.tight_layout()
    plt.show()
else:
    print('No usable categorical columns found for imbalance analysis.')
